In [2]:
# MNIST dataset으로 CNN 실습 - Sub classing
import tensorflow as tf
print(tf.__version__) # 2.20.0
import numpy as np
import matplotlib.pyplot as plt

# 1) 데이터 준비
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
# print(x_train[0])
print(x_train.shape) # (60000, 28, 28)

# 채널(channel) 추가 (흑백인 경우 1)
x_train = x_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0  # 정규화
x_test = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0

print(x_train.shape) # (60000, 28, 28, 1)  # 3차원 -> 4차원

2.20.0
(60000, 28, 28)
(60000, 28, 28, 1)


In [12]:
import tensorflow as tf

# 모델 정의
# 사용자 작성 class, func, layer 등을 keras 모델 저장/리로딩 시 자동으로 등록해주는 데코레이터
@tf.keras.utils.register_keras_serializable(package='custom')  # custom : 사용자 정의, losses : 손실함수, activation
class MyMnistCnn(tf.keras.Model):
  def __init__(self,**kwargs):
    super().__init__(**kwargs)
    # Conv Block 1
    self.conv1 = tf.keras.layers.Conv2D(filters=16, kernel_size=3, strides=1, padding='SAME', activation='relu')
    self.pool1 = tf.keras.layers.MaxPool2D(pool_size=(2,2))

    # Conv Block 2
    self.conv2 = tf.keras.layers.Conv2D(filters=32, kernel_size=3, strides=1, padding='SAME', activation='relu')
    self.pool2 = tf.keras.layers.MaxPool2D(pool_size=(2,2))

    # Fully Connected Layers
    self.flat = tf.keras.layers.Flatten()

    self.d1 = tf.keras.layers.Dense(units=64, activation='relu')
    self.drop1 = tf.keras.layers.Dropout(rate=0.2)
    self.d2 = tf.keras.layers.Dense(units=32, activation='relu')
    self.drop2 = tf.keras.layers.Dropout(rate=0.2)
    self.out = tf.keras.layers.Dense(units=10, activation='softmax')

  def call(self, inputs, training=False):
    x = self.conv1(inputs)
    x = self.pool1(x)
    x = self.conv2(x)
    x = self.pool2(x)
    x = self.flat(x)
    x = self.d1(x)
    x = self.drop1(x, training=training)
    x = self.d2(x)
    x = self.drop2(x, training=training)
    return self.out(x)

model = MyMnistCnn()
model.build(input_shape=(None, 28, 28, 1))
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'my_mnist_cnn_3', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Model: "my_mnist_cnn_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_10 (Conv2D)              │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [13]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

es = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
history = model.fit(x_train, y_train, epochs=100, batch_size=128, validation_split=0.1, callbacks=[es], verbose=1)

train_loss, train_acc = model.evaluate(x_train, y_train, verbose=0)
print(f"train_loss : {train_loss:.4f}, train_acc : {train_acc:.4f}")

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"test_loss : {test_loss:.4f}, test_acc : {test_acc:.4f}")

Epoch 1/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.8394 - loss: 0.5140 - val_accuracy: 0.9783 - val_loss: 0.0757
Epoch 2/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9573 - loss: 0.1490 - val_accuracy: 0.9857 - val_loss: 0.0468
Epoch 3/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9699 - loss: 0.1057 - val_accuracy: 0.9843 - val_loss: 0.0492
Epoch 4/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9759 - loss: 0.0850 - val_accuracy: 0.9872 - val_loss: 0.0460
Epoch 5/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9800 - loss: 0.0709 - val_accuracy: 0.9880 - val_loss: 0.0411
Epoch 6/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9827 - loss: 0.0604 - val_accuracy: 0.9897 - val_loss: 0.0348
Epoch 7/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9846 - loss: 0.0542 - val_accuracy: 0.9898 - val_loss: 0.0379
Epoch 8/100
422/422 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9866 - loss: 0.0465 - val_ac